# Qwen-Image-2.1 + ComfyUI + Cloudflare Quick Tunnel

Private interactive test without a Kaggle Dataset.

The notebook downloads ComfyUI, three official Qwen-Image-2.1 INT8 files, `cloudflared`, and two official UI workflows inside Kaggle:

- `Qwen Image 2.1 - Text to Image.json`
- `Qwen Image 2.1 - Image Edit.json`

The startup cell starts ComfyUI and the Quick Tunnel, prints the public URL, writes it to `/kaggle/working/tunnel_url.txt`, and finishes. The following hold cell keeps the kernel alive.

After the URL appears:

1. Open it in your browser.
2. In ComfyUI, open `Workflows`.
3. For editing, load `Qwen Image 2.1 - Image Edit.json`.
4. Upload one non-sensitive test image to `image_1` / its `LoadImage` node.
5. Run 2–3 generations and download the results.
6. STOP THE KAGGLE NOTEBOOK when finished.

## Session length

`SESSION_LIMIT_MINUTES` in the first code cell is the only knob: set it
before **Save & Run / Run All**. The last cell prints `REMAINING_MINUTES`
every five minutes and then stops ComfyUI and the tunnel cleanly instead of
waiting for the Kaggle wall. Use `0` to disable the timer.

Security: the Quick Tunnel URL is public and has no Cloudflare Access. Do not upload private or sensitive files during this test.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

WORKING = Path('/kaggle/working')
RUNTIME_ROOT = Path('/tmp/qwen-comfyui-interactive')
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
COMFY = RUNTIME_ROOT / 'ComfyUI'
USER_ROOT = COMFY / 'user'
COMFY_OUTPUT = WORKING / 'interactive_output'
RESULT_DIR = WORKING / 'interactive_test_result'
COMFY_LOG = RUNTIME_ROOT / 'comfyui_interactive.log'
TUNNEL_LOG = RUNTIME_ROOT / 'cloudflared_interactive.log'
STATUS_PATH = WORKING / 'interactive_test_status.json'
CLOUDFLARED = RUNTIME_ROOT / 'cloudflared'

COMFY_COMMIT = '93810483a4739a1588236919a3128d3070244146'
WORKFLOW_COMMIT = 'a7acaf8cee9ccccdaf4b26ce04a9fced4c4d13de'
WORKFLOW_BASE_URL = (
    'https://raw.githubusercontent.com/Comfy-Org/workflow_templates/'
    f'{WORKFLOW_COMMIT}/templates'
)
WORKFLOWS = {
    'Qwen Image 2.1 - Text to Image.json': f'{WORKFLOW_BASE_URL}/image_qwen_image_2_1_t2i.json',
    'Qwen Image 2.1 - Image Edit.json': f'{WORKFLOW_BASE_URL}/image_qwen_image_2_1_image_edit.json',
}
CLOUDFLARED_URL = (
    'https://github.com/cloudflare/cloudflared/releases/download/'
    '2026.9.3/cloudflared-linux-amd64'
)
MODEL_REPO = 'Comfy-Org/Qwen-Image-2.1'
MODEL_FILES = {
    'diffusion_models': 'diffusion_models/qwen_image_2.1_int8_convrot.safetensors',
    'text_encoders': 'text_encoders/qwen3vl_8b_int8_convrot.safetensors',
    'vae': 'vae/qwen_image_2.1_vae_bf16.safetensors',
}
MINIMUM_SIZES = {
    'diffusion_models': 7_000_000_000,
    'text_encoders': 9_000_000_000,
    'vae': 600_000_000,
}
KAGGLE_INPUT = Path('/kaggle/input')


def model_size_ok(target, folder):
    return target.is_file() and target.stat().st_size >= MINIMUM_SIZES[folder]


def find_model_root(base):
    # Locate a folder under base holding all three model files at full size.
    # Kaggle mount paths vary (/kaggle/input/<name>, /kaggle/input/datasets/<user>/<name>,
    # /kaggle/input/<name>/<name>), so a recursive search replaces a hard-coded list and
    # avoids silently re-downloading ~17 GB when the mount point differs.
    if not base.is_dir():
        return None
    for candidate in [base, *sorted(p for p in base.rglob('*') if p.is_dir())]:
        if all(model_size_ok(candidate / filename, folder)
               for folder, filename in MODEL_FILES.items()):
            return candidate
    return None


def find_model_file(filename, folder):
    # Find one valid model file anywhere under /kaggle/input, to reuse on fallback.
    if not KAGGLE_INPUT.is_dir():
        return None
    for path in KAGGLE_INPUT.rglob(Path(filename).name):
        if model_size_ok(path, folder):
            return path
    return None


MODEL_DATASET_ROOT = find_model_root(KAGGLE_INPUT)
MODEL_DATASET_AVAILABLE = MODEL_DATASET_ROOT is not None
MODELS_ROOT = MODEL_DATASET_ROOT if MODEL_DATASET_AVAILABLE else COMFY / 'models'
PORT = 8188
MAX_CLOUDFLARED_BYTES = 150 * 1024 * 1024

# --- SESSION TIME BUDGET: edit this before Save & Run / Run All ---
# How long the last cell keeps ComfyUI and the Quick Tunnel alive, in
# minutes. The countdown starts when that cell starts, so downloads and
# startup are not counted. Set it to 0 (or None) to hold until you stop
# the notebook yourself.
SESSION_LIMIT_MINUTES = 120
# ------------------------------------------------------------------
SCRIPT_STARTED = time.monotonic()

for path in (COMFY_OUTPUT, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)
COMFY_LOG.write_text('')
TUNNEL_LOG.write_text('')

status = {
    'stage': 'starting',
    'updated_at': datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds': 0,
    'comfy_commit': COMFY_COMMIT,
    'workflow_commit': WORKFLOW_COMMIT,
    'model_repo': MODEL_REPO,
    'model_files': MODEL_FILES,
    'models_root': str(MODELS_ROOT),
    'model_dataset_available': MODEL_DATASET_AVAILABLE,
    'port': PORT,
    'session_limit_minutes': SESSION_LIMIT_MINUTES,
    'tunnel_url': None,
    'workflow_names': list(WORKFLOWS),
    'comfy_ready': False,
    'tunnel_ready': False,
    'success': False,
}

def save_status(include_logs=False, announce=True):
    status['updated_at'] = datetime.now(timezone.utc).isoformat()
    status['elapsed_seconds'] = round(time.monotonic() - SCRIPT_STARTED, 2)
    if include_logs:
        status['comfy_log_tail'] = read_tail(COMFY_LOG, 12000)
        status['tunnel_log_tail'] = read_tail(TUNNEL_LOG, 8000)
    text = json.dumps(status, ensure_ascii=False, indent=2) + '\n'
    STATUS_PATH.write_text(text)
    (RESULT_DIR / 'interactive_test_status.json').write_text(text)
    if announce:
        print('STATUS', json.dumps(status, ensure_ascii=False), flush=True)

def read_tail(path, limit=12000):
    return path.read_text(errors='replace')[-limit:] if path.exists() else ''

def run(command, cwd=None, timeout=None):
    command = [str(part) for part in command]
    print('RUN', ' '.join(command), flush=True)
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )

def require_success(result, label):
    if result.returncode != 0:
        status['stage'] = f'failed: {label}'
        status[f'{label}_stdout'] = result.stdout[-8000:]
        status[f'{label}_stderr'] = result.stderr[-8000:]
        save_status(include_logs=True)
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')

save_status()


In [ ]:
def download_cloudflared():
    status['stage'] = 'downloading_cloudflared'
    save_status()
    part = CLOUDFLARED.with_suffix('.part')
    part.unlink(missing_ok=True)
    request = Request(CLOUDFLARED_URL, headers={'User-Agent': 'kaggle-comfyui-interactive-test/1.0'})
    total = 0
    with urlopen(request, timeout=60) as response, part.open('wb') as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk)
            if total > MAX_CLOUDFLARED_BYTES:
                raise RuntimeError('cloudflared download exceeded size limit')
            output.write(chunk)
    if total < 10 * 1024 * 1024:
        raise RuntimeError(f'cloudflared download is unexpectedly small: {total} bytes')
    with part.open('rb') as handle:
        if handle.read(4) != b'\x7fELF':
            raise RuntimeError('cloudflared is not an ELF executable')
    part.replace(CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
    status['cloudflared_size'] = CLOUDFLARED.stat().st_size
    save_status()

save_status()


In [ ]:
status['stage'] = 'cloning_comfyui'
save_status()
if not (COMFY / 'main.py').exists():
    require_success(run(['git', 'init', str(COMFY)]), 'git init')
    require_success(run(['git', '-C', str(COMFY), 'remote', 'add', 'origin', 'https://github.com/Comfy-Org/ComfyUI.git']), 'git remote add')
    require_success(run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', COMFY_COMMIT], timeout=900), 'git fetch ComfyUI')
    require_success(run(['git', '-C', str(COMFY), 'checkout', '--detach', 'FETCH_HEAD']), 'git checkout ComfyUI')

if not (COMFY / 'main.py').exists():
    raise RuntimeError('ComfyUI main.py was not found after checkout')

USER_ROOT.mkdir(parents=True, exist_ok=True)
status['stage'] = 'installing_comfyui_dependencies'
save_status()
require_success(run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input',
    '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub',
], timeout=1800), 'pip install ComfyUI dependencies')

download_cloudflared()
status['stage'] = 'comfyui_and_cloudflared_ready'
save_status()


In [ ]:
from huggingface_hub import hf_hub_download

WRITABLE_MODELS_ROOT = COMFY / 'models'
model_sizes = {}

if MODEL_DATASET_AVAILABLE:
    # Complete, valid dataset found: read straight from the read-only mount, no copy.
    MODELS_ROOT = MODEL_DATASET_ROOT
    status['stage'] = 'using_kaggle_dataset_models'
    status['model_source'] = 'kaggle_dataset'
    status['model_fallback_details'] = {}
    save_status()
    for folder, filename in MODEL_FILES.items():
        target = MODELS_ROOT / folder / Path(filename).name
        model_sizes[filename] = target.stat().st_size
        print('DATASET MODEL READY:', filename, target.stat().st_size, flush=True)
else:
    # Dataset missing or incomplete: build a writable models root. Valid files still
    # present under /kaggle/input are reused via symlink; the rest is downloaded.
    status['stage'] = 'preparing_fallback_qwen_int8_models'
    status['model_source'] = 'huggingface_fallback'
    status['model_fallback_details'] = {}
    save_status()
    WRITABLE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
    for folder, filename in MODEL_FILES.items():
        destination = WRITABLE_MODELS_ROOT / folder
        destination.mkdir(parents=True, exist_ok=True)
        target = destination / Path(filename).name
        source_kind = None
        if model_size_ok(target, folder):
            source_kind = 'cached'
            print('FALLBACK MODEL CACHED:', filename, target.stat().st_size, flush=True)
        else:
            reused = find_model_file(filename, folder)
            if reused is not None:
                target.unlink(missing_ok=True)
                try:
                    target.symlink_to(reused)
                except OSError:
                    target.unlink(missing_ok=True)
                if model_size_ok(target, folder):
                    source_kind = f'reused:{reused}'
                    print('REUSING DATASET FILE:', filename, '->', reused, flush=True)
                else:
                    target.unlink(missing_ok=True)
            if source_kind is None:
                print('DOWNLOADING', filename, flush=True)
                downloaded = Path(hf_hub_download(
                    repo_id=MODEL_REPO,
                    filename=filename,
                    local_dir=str(WRITABLE_MODELS_ROOT),
                ))
                if downloaded != target:
                    shutil.copy2(downloaded, target)
                source_kind = 'downloaded'
                print('DOWNLOADED', filename, target.stat().st_size, flush=True)
        status['model_fallback_details'][filename] = source_kind
        model_sizes[filename] = target.stat().st_size
        print('FALLBACK MODEL READY:', filename, target.stat().st_size, flush=True)
    MODELS_ROOT = WRITABLE_MODELS_ROOT

for folder, filename in MODEL_FILES.items():
    target = MODELS_ROOT / folder / Path(filename).name
    if not model_size_ok(target, folder):
        raise RuntimeError(f'Model verification failed: {target}')
    model_sizes[filename] = target.stat().st_size

status['models_root'] = str(MODELS_ROOT)
status['downloaded_model_sizes'] = model_sizes
status['stage'] = 'qwen_models_ready'
save_status()


In [ ]:
status['stage'] = 'downloading_official_ui_workflows'
save_status()

user_workflows = USER_ROOT / 'default' / 'workflows'
user_workflows.mkdir(parents=True, exist_ok=True)
workflow_info = {}

for workflow_name, workflow_url in WORKFLOWS.items():
    print('DOWNLOADING WORKFLOW', workflow_name, flush=True)
    request = Request(workflow_url, headers={'User-Agent': 'kaggle-qwen-comfyui-interactive-test/1.0'})
    with urlopen(request, timeout=120) as response:
        workflow_bytes = response.read()
    workflow = json.loads(workflow_bytes)
    if not isinstance(workflow.get('nodes'), list) or not workflow['nodes']:
        raise RuntimeError(f'{workflow_name} is not a usable ComfyUI UI workflow')
    if not isinstance(workflow.get('links'), list):
        raise RuntimeError(f'{workflow_name} does not contain UI links')

    local_workflow = user_workflows / workflow_name
    local_workflow.write_bytes(workflow_bytes)
    workflow_info[workflow_name] = {
        'source_url': workflow_url,
        'local_path': str(local_workflow),
        'bytes': len(workflow_bytes),
        'node_count': len(workflow['nodes']),
        'link_count': len(workflow['links']),
    }
    print('LOCAL WORKFLOW READY:', local_workflow, flush=True)

status['workflows'] = workflow_info
status['stage'] = 'ui_workflows_ready'
save_status()


In [ ]:
def stop_process(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)

comfy_process = None
tunnel_process = None
comfy_handle = COMFY_LOG.open('w', encoding='utf-8')
tunnel_handle = TUNNEL_LOG.open('w', encoding='utf-8')
known_pngs = set()
last_comfy_log_size = 0
status['generations'] = []

try:
    status['stage'] = 'starting_comfyui'
    save_status()
    comfy_process = subprocess.Popen([
        sys.executable, 'main.py',
        '--listen', '127.0.0.1',
        '--port', str(PORT),
        '--disable-auto-launch',
        '--lowvram',
        '--force-fp16',
        '--models-directory', str(MODELS_ROOT),
        '--user-directory', str(USER_ROOT),
        '--output-directory', str(COMFY_OUTPUT),
    ], cwd=str(COMFY), stdout=comfy_handle, stderr=subprocess.STDOUT, text=True)
    status['comfy_pid'] = comfy_process.pid

    deadline = time.time() + 240
    while time.time() < deadline:
        if comfy_process.poll() is not None:
            status['stage'] = 'failed_comfyui_exited'
            save_status(include_logs=True)
            raise RuntimeError('ComfyUI exited before becoming ready')
        try:
            with urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=3) as response:
                if response.status == 200:
                    status['comfy_ready'] = True
                    break
        except Exception:
            time.sleep(2)
    if not status['comfy_ready']:
        status['stage'] = 'failed_comfyui_timeout'
        save_status(include_logs=True)
        raise RuntimeError('ComfyUI did not become ready within 240 seconds')
    status['comfy_ready_at_seconds'] = round(time.monotonic() - SCRIPT_STARTED, 2)

    workflow_query = urlencode({'dir': 'workflows/', 'recurse': 'false'})
    with urlopen(f'http://127.0.0.1:{PORT}/userdata?{workflow_query}', timeout=10) as response:
        workflow_listing = json.loads(response.read())
    listed_names = {
        Path(item if isinstance(item, str) else item.get('path')).name
        for item in workflow_listing
        if isinstance(item, (str, dict))
        and (isinstance(item, str) or isinstance(item.get('path'), str))
    }
    missing_workflows = sorted(set(WORKFLOWS) - listed_names)
    if missing_workflows:
        status['stage'] = 'failed_workflows_not_listed'
        status['workflow_listing'] = workflow_listing
        save_status(include_logs=True)
        raise RuntimeError(f'ComfyUI did not list prepared workflows: {missing_workflows}')
    status['workflows_listed_by_comfyui'] = sorted(listed_names)
    save_status()

    status['stage'] = 'starting_cloudflared_quick_tunnel'
    save_status()
    tunnel_process = subprocess.Popen([
        str(CLOUDFLARED), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate',
    ], stdout=tunnel_handle, stderr=subprocess.STDOUT, text=True)
    status['tunnel_pid'] = tunnel_process.pid

    pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    deadline = time.time() + 180
    while time.time() < deadline:
        if tunnel_process.poll() is not None:
            status['stage'] = 'failed_cloudflared_exited'
            save_status(include_logs=True)
            raise RuntimeError('cloudflared exited before producing a URL')
        match = pattern.search(read_tail(TUNNEL_LOG, 30000))
        if match:
            status['tunnel_url'] = match.group(0)
            status['tunnel_ready'] = True
            break
        time.sleep(2)
    if not status['tunnel_ready']:
        status['stage'] = 'failed_cloudflared_timeout'
        save_status(include_logs=True)
        raise RuntimeError('Cloudflare Quick Tunnel did not produce a URL within 180 seconds')
    status['tunnel_ready_at_seconds'] = round(time.monotonic() - SCRIPT_STARTED, 2)

    public_url = status['tunnel_url']
    public_ready = False
    public_deadline = time.time() + 90
    while time.time() < public_deadline:
        try:
            with urlopen(public_url, timeout=10) as response:
                if response.status == 200:
                    public_ready = True
                    status['tunnel_http_status'] = response.status
                    break
        except Exception:
            time.sleep(2)
    if not public_ready:
        status['stage'] = 'failed_tunnel_http_timeout'
        save_status(include_logs=True)
        raise RuntimeError('Cloudflare Quick Tunnel URL did not return HTTP 200 within 90 seconds')

    tunnel_text = status['tunnel_url'] + '\n'
    (WORKING / 'tunnel_url.txt').write_text(tunnel_text)
    (RESULT_DIR / 'tunnel_url.txt').write_text(tunnel_text)

    status['stage'] = 'ready_for_browser_test'
    status['success'] = True
    status['message'] = (
        'Open the printed URL. Both Qwen text-to-image and image-edit workflows '
        'are available under Workflows. Test one reference image, then stop '
        'the Kaggle notebook.'
    )
    status['ready_at_seconds'] = round(time.monotonic() - SCRIPT_STARTED, 2)
    save_status(include_logs=True)

    print('', flush=True)
    print('=' * 72, flush=True)
    print('COMFYUI IS READY', flush=True)
    print('', flush=True)
    print('Open this URL in your browser:', flush=True)
    print(status['tunnel_url'], flush=True)
    print('', flush=True)
    print('Available workflows:', flush=True)
    for workflow_name in WORKFLOWS:
        print(f'  - {workflow_name}', flush=True)
    print('=' * 72, flush=True)
    print(f'TUNNEL_URL={status["tunnel_url"]}', flush=True)
    print('The next cell keeps the kernel alive.', flush=True)
except Exception:
    stop_process(tunnel_process)
    stop_process(comfy_process)
    comfy_handle.close()
    tunnel_handle.close()
    if not status['stage'].startswith('failed'):
        status['stage'] = 'failed_startup_exception'
    save_status(include_logs=True)
    raise


In [ ]:
print('SESSION HOLD: ComfyUI and the Quick Tunnel remain active.', flush=True)
print(f'TUNNEL_URL={status["tunnel_url"]}', flush=True)
last_url_announcement = time.monotonic()

hold_started = time.monotonic()
hold_deadline = (hold_started + SESSION_LIMIT_MINUTES * 60
                if SESSION_LIMIT_MINUTES else None)
last_countdown = hold_started
if hold_deadline is None:
    print('SESSION_LIMIT: disabled, runs until you stop the notebook.', flush=True)
else:
    print(f'SESSION_LIMIT: {SESSION_LIMIT_MINUTES} minutes from now.', flush=True)

try:
    while comfy_process.poll() is None and tunnel_process.poll() is None:
        if hold_deadline is not None:
            now = time.monotonic()
            if now >= hold_deadline:
                print('SESSION TIME LIMIT REACHED, stopping the session.', flush=True)
                break
            if now - last_countdown >= 300:
                last_countdown = now
                print('REMAINING_MINUTES',
                      round((hold_deadline - now) / 60, 1), flush=True)
        time.sleep(5)
        current_pngs = {
            str(path.relative_to(COMFY_OUTPUT))
            for path in COMFY_OUTPUT.rglob('*')
            if path.is_file() and path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}
        }
        for relative_path in sorted(current_pngs - known_pngs):
            path = COMFY_OUTPUT / relative_path
            status['generations'].append({
                'type': 'output_observed',
                'file': relative_path,
                'at_seconds': round(time.monotonic() - SCRIPT_STARTED, 2),
                'file_mtime': path.stat().st_mtime,
            })
            print('GENERATION OUTPUT OBSERVED:', relative_path, flush=True)
        known_pngs.update(current_pngs)

        if COMFY_LOG.exists():
            with COMFY_LOG.open('r', encoding='utf-8', errors='replace') as handle:
                handle.seek(last_comfy_log_size)
                log_delta = handle.read()
                last_comfy_log_size = handle.tell()
            for line in log_delta.splitlines():
                clean_line = re.sub(r'\x1b\[[0-9;]*m', '', line)
                if 'got prompt' in clean_line:
                    status['generations'].append({
                        'type': 'prompt_received',
                        'at_seconds': round(time.monotonic() - SCRIPT_STARTED, 2),
                    })
                    print('GENERATION PROMPT RECEIVED', flush=True)
                match = re.search(r'Prompt executed in ([0-9.]+) seconds', clean_line)
                if match:
                    status['generations'].append({
                        'type': 'prompt_executed',
                        'reported_seconds': float(match.group(1)),
                        'at_seconds': round(time.monotonic() - SCRIPT_STARTED, 2),
                    })
                    print('GENERATION PROMPT EXECUTED:', match.group(1), 'seconds', flush=True)

        status['stage'] = 'ready_for_browser_test'
        save_status(announce=False)

        if time.monotonic() - last_url_announcement >= 60:
            print(f'TUNNEL_URL={status["tunnel_url"]}', flush=True)
            last_url_announcement = time.monotonic()

    if (status['stage'] == 'ready_for_browser_test'
            and hold_deadline is not None
            and time.monotonic() >= hold_deadline):
        status['stage'] = 'session_time_limit_reached'
        status['success'] = True
        status['held_seconds'] = round(time.monotonic() - hold_started, 2)
        print(f'HELD_FOR_SECONDS {status["held_seconds"]}', flush=True)
        save_status(include_logs=True)
    else:
        status['stage'] = 'process_exited_unexpectedly'
        status['success'] = False
        save_status(include_logs=True)
        raise RuntimeError('ComfyUI or cloudflared exited while the session was active')
finally:
    stop_process(tunnel_process)
    stop_process(comfy_process)
    comfy_handle.close()
    tunnel_handle.close()
    if (not status['stage'].startswith('failed')
            and status['stage'] not in ('process_exited_unexpectedly',
                                        'session_time_limit_reached')):
        status['stage'] = 'stopped'
    save_status(include_logs=True)
